In [1]:
# Pin here for fix of CPU offloading bug; implemented in docker /uv
#%pip install "transformers==4.57.3" "accelerate==1.12.0" "bitsandbytes==0.49.1" "vllm<0.22.0"

In [2]:
# Environment fixes
import os
os.environ["FLASHINFER_DISABLE_VERSION_CHECK"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
from experiments_pretrained import *
from data import *
from config_record_activations import *
import pickle

In [4]:
# Set seeds
seed = 43
torch.manual_seed(seed);

In [5]:
# Get data: general prompts
dataset = get_data_mmlu(n_samples=n_samples, shuffle_seed=seed)
prompts = format_prompts_mmlu(dataset)

Streaming cais/mmlu (all) (samples: 15000)...


In [6]:
# Get model
model, tokenizer = load_model(model_id, enable_bnb=enable_bnb)
probe = MoEProbeQwen(model)

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

MoEHook: Scanning model for routers...
MoEHook: Attached probes to 24 router layers.
MoEHook: Model has 24 routers each with 60 experts and selects k=4 at each layer.
MoEProbe: Qwen1.5-MoE-A2.7B model also has shared expert with intermediate size: 5632 (Equivalent to ~4 routed experts)


In [ ]:
# Record activations over generalized MMLU questions
results = get_activations_mmlu(model, tokenizer, dataset, probe=probe, max_new_tokens=max_new_tokens, batch_size=batch_size)
#results = get_activations_mmlu(model, tokenizer, dataset, probe=probe, max_new_tokens=max_new_tokens, batch_size=1)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generating responses 1-64/14042...
Batch inference took 119.128s (1.861s/sample).
Generating responses 65-128/14042...
Batch inference took 116.392s (1.819s/sample).
Generating responses 129-192/14042...
Batch inference took 162.597s (2.541s/sample).
Generating responses 193-256/14042...
Batch inference took 155.776s (2.434s/sample).
Generating responses 321-384/14042...
Batch inference took 132.239s (2.066s/sample).
Generating responses 385-448/14042...
Batch inference took 115.969s (1.812s/sample).
Generating responses 449-512/14042...
Batch inference took 127.068s (1.985s/sample).
Generating responses 513-576/14042...
Batch inference took 113.490s (1.773s/sample).
Generating responses 577-640/14042...
Batch inference took 121.003s (1.891s/sample).
Generating responses 641-704/14042...
Batch inference took 118.526s (1.852s/sample).
Generating responses 705-768/14042...
Batch inference took 109.727s (1.714s/sample).
Generating responses 769-832/14042...
Batch inference took 161.015s (

In [ ]:
# Save results
with open(results_file, 'wb') as file:
    pickle.dump(results, file)